# Vision check (Nemotron Super VL)

Fetches the clearest true-colour Sentinel-2 image of a BARC field before and after harvest, and asks the vision model what it sees.
Extra evidence only: it never sets the label. Compare its hint with the BARC reference label below.

In [ ]:
# 1. Setup
import os, sys
from pathlib import Path
from fieldvision_agents.settings import REPO as ROOT
os.chdir(ROOT)

print('working in', Path.cwd())

In [ ]:
# 2. Offline tests (no credits)
from fieldvision_agents.vision import test_vision as t
for name, fn in vars(t).items():
    if name.startswith('test_'):
        fn(); print('PASS', name)

In [ ]:
# 3. Pick two silage and two grain BARC field-years
import geopandas as gpd
import pandas as pd
from fieldvision_agents.vision.vision import BARC_GPKG, BARC_LAYER
f = gpd.read_file(BARC_GPKG, layer=BARC_LAYER)
lab = f.melt(id_vars=['OBJECTID'], value_vars=['Corn_2021', 'Corn_2022', 'Corn_2023'], var_name='col', value_name='label')
lab = lab[lab['label'].isin(['Silage', 'Grain'])].assign(year=lambda d: d['col'].str[-4:].astype(int))
sample = pd.concat([lab[lab['label'] == 'Silage'].head(2), lab[lab['label'] == 'Grain'].head(2)])
sample[['OBJECTID', 'year', 'label']]

In [ ]:
# 4. Run the vision check on the sample (4 fields x 1 Super VL call)
from IPython.display import Image, display
from fieldvision_agents.vision.vision import check_barc_field
for _, row in sample.iterrows():
    verdict, before, after = check_barc_field(int(row['OBJECTID']), int(row['year']))
    print(f"BARC {row['OBJECTID']} {row['year']}  reference={row['label']}  vision hint={verdict['hint']}", verdict)
    display(Image(before), Image(after))